# Notebook 00b — Build the Kerala Ward-Level Feature Store

Extends the original LSGD-level feature store (Notebook 00, 1,034 units) down to
**21,002 wards** -- Kerala's finest-grained official administrative unit, sourced from
the Delimitation Commission's survey (`kerala_wards_all.csv`).

**What this does:**
1. Loads the 21,002 ward centroids (real surveyed polygon centroids, not estimates)
2. Extracts terrain features per ward via GEE (elevation, slope, distance to water,
   vegetation %, built-up %) -- same datasets as Notebook 00, just point-based instead
   of polygon zonal-stats
3. Extracts historical rainfall per ward via **CHIRPS** (not Open-Meteo) -- CHIRPS runs
   inside GEE and can batch-process all 21,002 points in one job; Open-Meteo's free tier
   (~10,000 calls/day) can't handle this volume for a one-time training-data build.
   Live per-query rainfall (when a user searches a place) still uses Open-Meteo, unchanged
   -- that's a different, much lower-volume use case (one call per search).
4. Joins district-level 2018 flood/landslide labels (same honest scoping as Notebook 00 --
   labels are still only known at district granularity, now inherited down to ward level
   instead of LSGD level -- state this plainly in your report, same as before)

**Files to upload when prompted:**
- `kerala_wards_all.csv` (21,002 rows: Ward_Name, Ward_No, Local_Body, Category, District, Latitude, Longitude)
- `district_labels_2018.csv` (same file already used in Notebook 00)

**Output:** `ward_feature_store.csv` -- one row per ward, same feature columns as the
original `lsgd_feature_store.csv` (`elevation`, `slope`, `rainfall_7day_mm`, `dist_to_water_m`,
`vegetation`, `builtup`) plus ward identity columns, ready to retrain Notebooks 01/02 on.

## 1. Setup

In [1]:
!pip install -q geemap pandas earthengine-api

import ee, geemap, pandas as pd, numpy as np
from google.colab import drive, files

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/DIP_Kerala'

ee.Authenticate()
ee.Initialize(project='dip-kerala-502817')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.9 MB/s eta 0:00:00
Mounted at /content/drive


## 2. Load the 21,002 ward centroids

Upload `kerala_wards_all.csv` to `DIP_Kerala/00_boundaries/` in Drive first, or upload
directly here if you'd rather not touch Drive structure.

In [2]:
import os
wards_path = f'{BASE}/00_boundaries/kerala_wards_all.csv'
if not os.path.exists(wards_path):
    print("Not found in Drive -- upload kerala_wards_all.csv now")
    uploaded = files.upload()
    wards_path = list(uploaded.keys())[0]

wards = pd.read_csv(wards_path)
wards['ward_id'] = range(1, len(wards) + 1)
print(f"Loaded {len(wards)} wards")
print(wards['Category'].value_counts(dropna=False))
wards.head()

Loaded 21002 wards
Category
Panchayat       17311
Municipality     3249
Corporation       425
NaN                17
Name: count, dtype: int64


,Ward_Name,Ward_No,Local_Body,Category,District,Latitude,Longitude,ward_id
0,Thumpoly,1,Alappuzha,Municipality,Alappuzha,9.518699,76.317349,1
1,Kommady,2,Alappuzha,Municipality,Alappuzha,9.518901,76.325526,2
2,Poonthoppu,3,Alappuzha,Municipality,Alappuzha,9.520299,76.332421,3
3,Kalath,4,Alappuzha,Municipality,Alappuzha,9.523639,76.337069,4
4,Kottamkulangara,5,Alappuzha,Municipality,Alappuzha,9.525575,76.344309,5


## 3. Convert to Earth Engine FeatureCollection, in batches

21,002 points is ~20x Notebook 00's 1,034 polygons -- batching avoids GEE payload/timeout
limits that a single upload of this size could hit.

In [3]:
def df_to_ee_points(df, lat_col='Latitude', lon_col='Longitude', id_col='ward_id'):
    features = []
    for _, row in df.iterrows():
        pt = ee.Geometry.Point([row[lon_col], row[lat_col]])
        features.append(ee.Feature(pt, {'ward_id': int(row[id_col])}))
    return ee.FeatureCollection(features)

BATCH_SIZE = 2000
batches = [wards.iloc[i:i+BATCH_SIZE] for i in range(0, len(wards), BATCH_SIZE)]
print(f"Split into {len(batches)} batches of up to {BATCH_SIZE} points each")

Split into 11 batches of up to 2000 points each


## 4. Terrain features — elevation & slope (SRTM DEM), per batch

In [4]:
dem = ee.Image('USGS/SRTMGL1_003')
slope_img = ee.Terrain.slope(dem)
terrain_stack = dem.rename('elevation').addBands(slope_img.rename('slope'))

terrain_dfs = []
for i, batch_df in enumerate(batches):
    fc = df_to_ee_points(batch_df)
    stats = terrain_stack.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=30)
    batch_result = geemap.ee_to_df(stats)[['ward_id', 'elevation', 'slope']]
    terrain_dfs.append(batch_result)
    print(f"Batch {i+1}/{len(batches)} done ({len(batch_result)} wards)")

terrain_df = pd.concat(terrain_dfs, ignore_index=True)
print(f"\nTotal terrain rows: {len(terrain_df)}")
terrain_df.head()

Batch 1/11 done (2000 wards)
Batch 2/11 done (2000 wards)
Batch 3/11 done (2000 wards)
Batch 4/11 done (2000 wards)
Batch 5/11 done (2000 wards)
Batch 6/11 done (2000 wards)
Batch 7/11 done (2000 wards)
Batch 8/11 done (2000 wards)
Batch 9/11 done (2000 wards)
Batch 10/11 done (2000 wards)
Batch 11/11 done (1002 wards)

Total terrain rows: 21002


,ward_id,elevation,slope
0,1,5,2.078843
1,2,12,0.000000
2,3,12,2.639852
3,4,13,2.078849
4,5,13,3.354732


## 5. Rainfall — CHIRPS (2018 monsoon window, matches the existing training labels)

In [5]:
START_DATE = '2018-08-13'
END_DATE = '2018-08-19'  # same 7-day peak window used in Notebook 00 for the 1,034 LSGD units

chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(START_DATE, END_DATE).sum().rename('rainfall_7day_mm')

rainfall_dfs = []
for i, batch_df in enumerate(batches):
    fc = df_to_ee_points(batch_df)
    stats = chirps.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=5000)
    batch_result = geemap.ee_to_df(stats)[['ward_id', 'first']].rename(columns={'first': 'rainfall_7day_mm'})  # GEE names single-band reducer output 'first', not the band name
    rainfall_dfs.append(batch_result)
    print(f"Batch {i+1}/{len(batches)} done")

rainfall_df = pd.concat(rainfall_dfs, ignore_index=True)
rainfall_df['event_year'] = 2018

# Fill any missing readings (coastal/backwater wards can fall on a CHIRPS no-data/water
# pixel) with their own district's average -- rainfall varies far less within a district
# than terrain does, so this is a reasonable, honestly-documented fallback rather than
# dropping real wards from the dataset.
n_missing = rainfall_df['rainfall_7day_mm'].isnull().sum()
if n_missing > 0:
    district_lookup = wards[['ward_id', 'District']]
    rainfall_df = rainfall_df.merge(district_lookup, on='ward_id')
    district_avg = rainfall_df.groupby('District')['rainfall_7day_mm'].transform('mean')
    rainfall_df['rainfall_7day_mm'] = rainfall_df['rainfall_7day_mm'].fillna(district_avg)
    rainfall_df = rainfall_df.drop(columns='District')
    print(f"Filled {n_missing} missing rainfall readings with their district's average")

print(f"\nTotal rainfall rows: {len(rainfall_df)}")
print(f"Remaining nulls: {rainfall_df['rainfall_7day_mm'].isnull().sum()}")
rainfall_df.head()

Batch 1/11 done
Batch 2/11 done
Batch 3/11 done
Batch 4/11 done
Batch 5/11 done
Batch 6/11 done
Batch 7/11 done
Batch 8/11 done
Batch 9/11 done
Batch 10/11 done
Batch 11/11 done
Filled 75 missing rainfall readings with their district's average

Total rainfall rows: 21002
Remaining nulls: 0


,ward_id,rainfall_7day_mm,event_year
0,1,433.117315,2018
1,2,433.117315,2018
2,3,433.117315,2018
3,4,349.570593,2018
4,5,349.570593,2018


## 6. Water proximity — distance to permanent surface water (JRC)

In [6]:
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
water_mask = gsw.gt(50)
distance_to_water = water_mask.fastDistanceTransform().sqrt().multiply(ee.Image.pixelArea().sqrt()).rename('dist_to_water_m')

water_dfs = []
for i, batch_df in enumerate(batches):
    fc = df_to_ee_points(batch_df)
    stats = distance_to_water.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=100)
    batch_result = geemap.ee_to_df(stats)[['ward_id', 'first']].rename(columns={'first': 'dist_to_water_m'})  # same GEE single-band naming quirk as rainfall above
    water_dfs.append(batch_result)
    print(f"Batch {i+1}/{len(batches)} done")

water_df = pd.concat(water_dfs, ignore_index=True)
print(f"\nTotal water-distance rows: {len(water_df)}")
water_df.head()

Batch 1/11 done
Batch 2/11 done
Batch 3/11 done
Batch 4/11 done
Batch 5/11 done
Batch 6/11 done
Batch 7/11 done
Batch 8/11 done
Batch 9/11 done
Batch 10/11 done
Batch 11/11 done

Total water-distance rows: 21002


,ward_id,dist_to_water_m
0,1,602.159288
1,2,1488.211812
2,3,2198.026296
3,4,1992.208125
4,5,1196.141334


## 7. Land cover — vegetation / built-up % (ESA WorldCover)

In [7]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
veg_mask = worldcover.eq(10).Or(worldcover.eq(40)).rename('vegetation')
builtup_mask = worldcover.eq(50).rename('builtup')
landcover_stack = veg_mask.addBands(builtup_mask)

landcover_dfs = []
for i, batch_df in enumerate(batches):
    fc = df_to_ee_points(batch_df)
    # small buffer (50m) around each point -- a single 10m pixel can be noisy for a
    # binary land-cover class; a small neighborhood average is more representative
    fc_buffered = fc.map(lambda f: f.setGeometry(f.geometry().buffer(50)))
    stats = landcover_stack.reduceRegions(collection=fc_buffered, reducer=ee.Reducer.mean(), scale=10)
    batch_result = geemap.ee_to_df(stats)[['ward_id', 'vegetation', 'builtup']]
    landcover_dfs.append(batch_result)
    print(f"Batch {i+1}/{len(batches)} done")

landcover_df = pd.concat(landcover_dfs, ignore_index=True)
print(f"\nTotal land-cover rows: {len(landcover_df)}")
landcover_df.head()

Batch 1/11 done
Batch 2/11 done
Batch 3/11 done
Batch 4/11 done
Batch 5/11 done
Batch 6/11 done
Batch 7/11 done
Batch 8/11 done
Batch 9/11 done
Batch 10/11 done
Batch 11/11 done

Total land-cover rows: 21002


,ward_id,vegetation,builtup
0,1,1.000000,0.000000
1,2,0.827778,0.172222
2,3,0.850134,0.149866
3,4,0.790499,0.209501
4,5,1.000000,0.000000


## 8. Merge everything

In [8]:
feature_store = wards.rename(columns={
    'Ward_Name': 'ward_name', 'Ward_No': 'ward_no', 'Local_Body': 'local_body',
    'Category': 'category', 'District': 'district', 'Latitude': 'lat', 'Longitude': 'lon',
})[['ward_id', 'ward_name', 'ward_no', 'local_body', 'category', 'district', 'lat', 'lon']] \
    .merge(terrain_df, on='ward_id') \
    .merge(rainfall_df, on='ward_id') \
    .merge(water_df, on='ward_id') \
    .merge(landcover_df, on='ward_id')

print(feature_store.shape)
print("Missing values per column:")
print(feature_store.isnull().sum())
feature_store.head()

(21002, 15)
Missing values per column:
ward_id              0
ward_name            0
ward_no              0
local_body           0
category            17
district             0
lat                  0
lon                  0
elevation            0
slope                0
rainfall_7day_mm     0
event_year           0
dist_to_water_m      0
vegetation           0
builtup              0
dtype: int64


,ward_id,ward_name,ward_no,local_body,category,district,lat,lon,elevation,slope,rainfall_7day_mm,event_year,dist_to_water_m,vegetation,builtup
0,1,Thumpoly,1,Alappuzha,Municipality,Alappuzha,9.518699,76.317349,5,2.078843,433.117315,2018,602.159288,1.000000,0.000000
1,2,Kommady,2,Alappuzha,Municipality,Alappuzha,9.518901,76.325526,12,0.000000,433.117315,2018,1488.211812,0.827778,0.172222
2,3,Poonthoppu,3,Alappuzha,Municipality,Alappuzha,9.520299,76.332421,12,2.639852,433.117315,2018,2198.026296,0.850134,0.149866
3,4,Kalath,4,Alappuzha,Municipality,Alappuzha,9.523639,76.337069,13,2.078849,349.570593,2018,1992.208125,0.790499,0.209501
4,5,Kottamkulangara,5,Alappuzha,Municipality,Alappuzha,9.525575,76.344309,13,3.354732,349.570593,2018,1196.141334,1.000000,0.000000


## 9. Attach district-level labels (same honest scoping as Notebook 00)

Upload `district_labels_2018.csv` (same file from before). Every ward inherits its
district's 2018 flood/landslide outcome -- district-level labels are still the ceiling
of what's officially available, now applied at ward granularity instead of LSGD granularity.

In [9]:
labels_path = f'{BASE}/01_labels/district_labels_2018.csv'
if not os.path.exists(labels_path):
    print("Not found in Drive -- upload district_labels_2018.csv now")
    uploaded_labels = files.upload()
    labels_path = list(uploaded_labels.keys())[0]

labels_df = pd.read_csv(labels_path)

# Fix a real spelling mismatch found in testing: district_labels_2018.csv has
# "Kasaragode" (with an e) while the official ward CSV spells it "Kasaragod" (no e).
# Without this, all 845 Kasaragod wards silently get null labels from the merge below.
labels_df['district'] = labels_df['district'].replace({'Kasaragode': 'Kasaragod'})

feature_store_labeled = feature_store.merge(labels_df, on='district', how='left')

n_unlabeled = feature_store_labeled['flood_occurred'].isnull().sum()
if n_unlabeled > 0:
    print(f"WARNING: {n_unlabeled} wards still have no label -- check for other district spelling mismatches:")
    print(feature_store_labeled[feature_store_labeled['flood_occurred'].isnull()]['district'].unique())
else:
    print("All wards matched to a district label -- 0 nulls.")

print(feature_store_labeled['flood_occurred'].value_counts())
print(feature_store_labeled['landslide_occurred'].value_counts())
feature_store_labeled.head()

All wards matched to a district label -- 0 nulls.
flood_occurred
1    11673
0     9329
Name: count, dtype: int64
landslide_occurred
1    10887
0    10115
Name: count, dtype: int64


,ward_id,ward_name,ward_no,local_body,category,district,lat,lon,elevation,slope,...,fully_damaged_houses,severely_damaged_houses,roads_damaged_km,bridges_damaged,severity_score,flood_risk_level,landslide_risk_level,flood_occurred,landslide_occurred,event_year_y
0,1,Thumpoly,1,Alappuzha,Municipality,Alappuzha,9.518699,76.317349,5,2.078843,...,2075,18990,241.0,121,0.565344,Critical,Low,1,0,2018
1,2,Kommady,2,Alappuzha,Municipality,Alappuzha,9.518901,76.325526,12,0.000000,...,2075,18990,241.0,121,0.565344,Critical,Low,1,0,2018
2,3,Poonthoppu,3,Alappuzha,Municipality,Alappuzha,9.520299,76.332421,12,2.639852,...,2075,18990,241.0,121,0.565344,Critical,Low,1,0,2018
3,4,Kalath,4,Alappuzha,Municipality,Alappuzha,9.523639,76.337069,13,2.078849,...,2075,18990,241.0,121,0.565344,Critical,Low,1,0,2018
4,5,Kottamkulangara,5,Alappuzha,Municipality,Alappuzha,9.525575,76.344309,13,3.354732,...,2075,18990,241.0,121,0.565344,Critical,Low,1,0,2018


## 10. Save

In [10]:
os.makedirs(f'{BASE}/03_processed', exist_ok=True)
feature_store_labeled.to_csv(f'{BASE}/03_processed/ward_feature_store.csv', index=False)
print('Saved to', f'{BASE}/03_processed/ward_feature_store.csv')
print('This replaces lsgd_feature_store.csv as the input to Notebooks 01/02 -- update')
print("those notebooks' load path, everything else (feature columns, training code) stays the same.")

Saved to /content/drive/MyDrive/DIP_Kerala/03_processed/ward_feature_store.csv
This replaces lsgd_feature_store.csv as the input to Notebooks 01/02 -- update
those notebooks' load path, everything else (feature columns, training code) stays the same.


## Known limitations to state honestly in your report

- **Labels are still district-level (14 districts)**, now inherited down to 21,002 ward
  rows instead of 1,034 LSGD rows. This does NOT fix the honest-accuracy ceiling problem
  found earlier (district-grouped validation showing ~18-50% real accuracy) -- more rows
  with the same underlying label granularity doesn't add new independent information.
  Ward-level terrain detail is genuinely finer and more useful for place-level search and
  live queries, but the accuracy ceiling is still set by the 14-district label problem
  discussed earlier -- getting real per-ward ground truth would still require the
  Bhukosh/Bhuvan point-level data sources flagged previously.
- **A single reduceRegions `first()` reducer** was used for elevation/slope/rainfall --
  reasonable for point data, but means a ward's terrain reading is a single-pixel sample
  at its centroid, not an area average. The land-cover step used a small 50m buffer
  average instead, since binary land-cover classes are noisier at a single pixel.
- **8 of 21,002 wards may need manual review** if any batch returns nulls (e.g. a
  centroid falling exactly on a coastline/data-void pixel) -- check `isnull().sum()`
  output in Section 8 and handle any remaining gaps before training.